# rpart 教程（1/3）：建树与剪枝机制

 >  本文件在系列中的位置 ：这是 rpart 系列教程的 第 1 部分 ，承接  `00_总览与环境.ipynb`  的预备知识，集中讲透 所有 method 共用的核心机制 ：不纯度、improve、cp、cp 表、1-SE 规则、三棵树对比。
 >
 >  系列结构 ：
 > -  `00_总览与环境.ipynb` ：预备 + 附录 A（predict 速查）+ 附录 B（常见坑）
 > -  本文件（1） ：建树与剪枝机制 +  附录 A（Breiman 排序定理证明）  +  附录 B（1-SE 深探） 
 > -  `02_按method展开.ipynb` ：class / anova / poisson / 生存 + 附录 C（其余函数与数据集）
 > -  `03_横切专题.ipynb` ：代价敏感、surrogate、重要性、复杂度
 >
 >  前置知识 ：需要先读过  `00_总览与环境.ipynb`  的 0.4 术语表、0.5 控制参数、0.6 1-SE 规则与  `prune_1se()` 。

---

## 本文件导读

本文件用二维平面上的分类/回归直观例子，回答四个问题：树怎么分段拟合？分支依据是什么？cp 如何控制复杂度？cp 表怎么读、1-SE 怎么用？

1.  1.1 直观引入 ：曲线分类边界 + 平衡数据，看轴平行分割如何逼近曲线；
2.  1.2 分支原理 ：Gini/MSE 不纯度、候选切点搜索、 无序离散变量的排序定理 ；
3.  1.3 cp 定义 ：代价复杂度 $R_\alpha(T)=R(T)+\alpha|T|$，建树/剪枝两处作用；
4.  1.4 cp 表逐列解读 ：弱链接剪枝、$g(t)$、CP 列的来历、五列含义与读法；
5.  1.5 三棵树对比 ：完全树 / 最简树 / 1-SE 最优树，边界图 + 误差汇总；
6.  附录 A ： Breiman 类别排序定理的详细证明 （紧接 1.2 节，解释为何只需排序后相邻切分）；
7.  附录 B ： 1-SE 规则的深入探讨 （统计学逻辑 + 生活化比喻）。 

# 第 1 节 建树与剪枝机制 

本节用二维平面上的分类/回归直观例子引入：树如何分段拟合、分支的依据是什么、cp 如何控制复杂度、cp 表怎么读、1-SE 规则怎么用。这些是所有 method 共用的，第 2 节各 method 不再重复。 

## 1.1 直观引入：树能分段拟合非线性边界 

构造一个两类分类问题：真实分类界限为曲线 $x_2 = x_1\cos(x_1)$，两类分别在曲线上下各偏移 4 并加噪声。决策树用 轴平行 的分割逼近这条曲线。

 >  教学点设计：干净、平衡的类别场景 。这里我们有意让两类各生成 10000 个样本，保证 绝对的类别平衡 。在平衡数据下，rpart 的默认行为（如 0.5 的概率阈值、Gini 不纯度自然倾向于均等划分）是 完全无偏且高效 的。此时模型不需要处理“倾向多数类”的纠偏问题，它的 全部关注点都在于如何纯粹、高效地切分特征空间 。这为后续理解分支原理（Gini、improve）提供了最纯粹的背景。代码中将直接输出两类样本的频数与比例，用数据印证这一设计。

 > 本块自带数据生成与绘图，独立可运行。 

以下代码 生成用于后续建树和可视化的模拟数据 （包括训练集和测试集），并绘制训练集散点图。代码中直接输出了两类样本的频数与比例，以印证“绝对平衡”的设计。 

In [ ]:
options(warn=-1)                                     # 忽略一切警告
options('width'=140)                                 # 充分利用打印宽度
options(repr.plot.width=15, repr.plot.height=10)     # 满幅绘图
library(rpart)                                       # 加载 rpart 包（最小依赖）
n   <- 10000                                           # 训练/测试集各自的样本量（两类各 n 个，保证平衡）
u   <- 10                                              # 自变量 x1 的取值范围 [-10, 10]
v   <- 5                                               # 噪声强度（正态噪声标准差）

# -------------------- 构建训练集 sim_cl_tr --------------------
set.seed(11)                                         # 固定随机种子，保证结果可复现
x1   <- sort(runif(n, min = -u, max = u))              # 生成 n 个均匀分布的 x1，并从小到大排序
x2   <- x1 * cos(x1) + v * rnorm(n) - 4                # 按理论曲线生成 x2，向下偏移 4，加入正态噪声
d0   <- data.frame(y = 1, x1, x2)                      # 组装第一类数据（y = 1）

set.seed(22)                                         # 固定随机种子
x1   <- sort(runif(n, min = -u, max = u))              # 重新生成 x1
x2   <- x1 * cos(x1) + v * rnorm(n) + 4                # 按理论曲线生成 x2，向上偏移 4，加入正态噪声
d1   <- data.frame(y = 2, x1, x2)                      # 组装第二类数据（y = 2）

sim_cl_tr   <- rbind(d0, d1)                               # 合并两类数据，共 2n 行（完美平衡）
sim_cl_tr$y   <- factor(sim_cl_tr$y)                           # 目标变量转因子，以便按分类树建模

# -------------------- 输出平衡性验证 --------------------
table(sim_cl_tr$y)                                       # 两类绝对样本量（应为各 10000）
prop.table(table(sim_cl_tr$y))                           # 两类比例（应为各 0.5）

# -------------------- 绘制训练集散点图 --------------------
plot(sim_cl_tr$x1, sim_cl_tr$x2, col = sim_cl_tr$y,              # 画散点图，按类别着色
     xlab =  "x1", ylab =  "x2", cex = 0.5)           # 设置坐标轴标签，点调小防重叠
curve(x * cos(x), col =  "red", lwd = 6, lty = 3, add = TRUE)  # 叠加真实分类界限
legend( "topright",  "理论分类界限",                    # 图例放在右上角，避免遮挡
       lwd = 6, col =  "red", lty = 3, cex = 0.8)     # 图例线型、颜色、字号
grid()                                               # 添加网格线

# -------------------- 构建测试集 sim_cl_te --------------------
set.seed(33)                                         # 固定随机种子
x1   <- sort(runif(n, min = -u, max = u))              # 生成 x1
x2   <- x1 * cos(x1) + v * rnorm(n) - 4                # 生成第一类 x2
d0   <- data.frame(y = 1, x1, x2)                      # 组装第一类数据

set.seed(44)                                         # 固定随机种子
x1   <- sort(runif(n, min = -u, max = u))              # 生成 x1
x2   <- x1 * cos(x1) + v * rnorm(n) + 4                # 生成第二类 x2
d1   <- data.frame(y = 2, x1, x2)                      # 组装第二类数据

sim_cl_te   <- rbind(d0, d1)                               # 合并测试集（同样平衡）
sim_cl_te$y   <- factor(sim_cl_te$y)                           # 目标变量转因子 

## 1.2 建树的核心原理：分支的依据是什么？

决策树在每一个节点上，都会面临一个核心问题： “我应该在哪个变量的哪个值上切一刀，才能让切出来的两拨人最纯粹？” 

为了衡量“纯粹程度”，CART 算法引入了 不纯度（Impurity） 的概念：

-  分类树 ：使用 Gini 指数 。$G = 1 - \sum p_k^2$。如果节点里只有一类，$G=0$（最纯）；如果两类各占 50%，$G=0.5$（最杂）。
-  回归树 ：使用 方差（MSE） 。如果节点里所有 $y$ 值都一样，方差为 0（最纯）。

### 寻找最优切分点

在确定当前节点怎么切时，由于 rpart 是严格的 二叉决策树 （每次一刀只能切成两份），候选切点的搜索逻辑因变量类型而异：

1.  遍历当前节点所有可能的分裂变量 （如 $x_1, x_2, \text{Type}$）。
2.  生成候选切点并计算 ：
   *  连续变量（精确穷举） ：将节点内该变量的取值去重并排序，得到唯一值序列 $v_1  < v_2  < \dots  < v_m$。算法取相邻两个唯一值的 中点  $c_i = \frac{v_i + v_{i+1}}{2}$ 作为候选切点（条件为 $x  < c_i$）。切在区间 $(v_i, v_{i+1})$ 内任何位置划分效果均相同，因此尝试这 $m-1$ 个中点就等同于 彻底穷举 了所有可能的二分类法。借助排序，该过程能在 $O(n \log n)$ 内完成。
   *  无序离散变量（基于定理的降维搜索） ：若变量有 $K$ 个无序类别，数学上要找出最佳的两组划分，理论上需要穷举 $2^{K-1} - 1$ 种组合。然而在 二分类问题且使用 Gini 不纯度 （或回归使用 MSE）时，Breiman 等人证明了一个重要的定理： 按类别内目标变量均值（或比例）排序后，在相邻类别间寻找切点，所得结果与穷举所有 $2^{K-1}-1$ 种组合等价 。
     *  具体做法 ：设类别为 $c_1,\dots,c_K$，计算每个类别的正类比例 $p_k = P(y=1\mid X=c_k)$（或回归中的均值）。按 $p_k$ 从小到大排列，得到 $p_{(1)}\le p_{(2)}\le \cdots\le p_{(K)}$。只在这个排好的序列的 相邻缝隙 处切分（即左支取前 $k$ 个类别，右支取后 $K-k$ 个类别），共 $K-1$ 种候选切法。
     *  关键 ：跨越式组合（如把 $p$ 值一高一低的两个类别分到同一侧）在数学上 永远不可能 比相邻缝隙切分取得更低的 Gini。因此，排序法在 Gini/MSE 下是 精确最优 的。 严格证明见本文件附录 A 。
     *  适用边界 ：该定理 仅对二分类 + Gini 或回归 + MSE 成立 。若使用信息增益（Entropy）或多分类问题（响应变量 $ >2$ 类），排序定理失效，此时排序法只是一种 启发式近似 。rpart 官方文档明确说明：使用信息增益时需要不同的算法，不能使用类别排序的捷径。
   *  有序离散变量（保持顺序） ：若是有序因子（如“低  < 中  < 高”），切点必须保持原有顺序，只能在序列的缝隙中切分（前 $k$ 个类在左，后 $K-k$ 个类在右）。
3.  计算切分后的加权不纯度 ：对于每一种候选切分，计算左右子节点的加权不纯度：
   \[
   \text{Imp}_{\text{new}} = \frac{n_L}{n} G_L + \frac{n_R}{n} G_R
   \]
4.  计算不纯度下降量（Improve） ：$\text{Improve} = \text{父节点不纯度} - \text{切分后子节点加权不纯度}$。
5.  选择最优分支 ：遍历完当前节点的所有变量及候选切点后， 选择使 Improve 最大的那个切点 。

 >  注意：多分类与特殊准则下的“组合爆炸”陷阱 
 > 上面提到的无序变量排序捷径 仅在二分类或回归（使用 Gini/MSE）时保证最优 。如果是 多分类问题 （响应变量类别数 $ >2$）或使用了其他分裂准则（如 Twoing、Entropy），该定理不再成立。此时要找到最优切分，理论上确实只能穷举所有组合，这会导致计算量呈指数级爆炸。在实际应用中，如果无序类别极多（如 ID、邮编），通常建议预处理剔除或转为数值特征，否则不仅树容易过拟合，计算也会极其缓慢。 

## 1.3 复杂度参数 cp：定义、计算与作用 

 cp（complexity parameter，复杂度参数） 是 rpart 控制树复杂度的核心机制，本质是 CART 的 代价复杂度剪枝（cost-complexity pruning） 中的惩罚系数 $\alpha$（即公式里的 $\alpha$ 就是 rpart 中的 cp）。

### cp 的定义

对一棵树 $T$，定义其 代价复杂度 为：

$$
R_\alpha(T) = R(T) + \alpha |T|
$$

其中：
- $R(T)$ 是树 $T$ 的训练误差（分类用误判率或 Gini，回归用 SSE）；
- $|T|$ 是树的 叶子节点数 （等价于分裂数 + 1，衡量复杂度）；
- $\alpha \ge 0$ 是惩罚系数，即 rpart 中的 cp 。

$\alpha$ 越大，对复杂树的惩罚越重，越倾向于选简单的树；$\alpha=0$ 时只看训练误差，会选最复杂的树（充分生长）。

 直观地看 ：一定 $\alpha$ 下，比较剪枝前和剪枝后的 $R_\alpha(T)$，若剪枝后更低，则实施剪枝。这为我们提供了选择子树的数学总纲。



### cp 如何控制复杂度

cp 在 rpart 中有 两处 控制作用：

 (1) 建树阶段 ： `rpart.control(cp = ...)`  作为 生长停止阈值 。在每次尝试分割时，要求这次分割带来的误差下降量（improve）至少达到  `cp × 根节点误差` ，否则不分裂。默认  `cp = 0.01`  意味着：一次分裂必须让整体误差相对下降至少 1%，否则不值得。 `cp = 0`  则取消该限制，树会充分生长。

 (2) 剪枝阶段 ： `prune(tree, cp = ...)`  从已建好的树中提取对应 cp 值的子树。由于弱链接剪枝生成的子树是 嵌套的 ，给定任意 cp，都可以在 cp 表中找到「当前 cp 对应的那棵子树」并返回。

 cp 与树复杂度的关系（一句话） ：cp 越大 → 惩罚越重 → 树越简单（nsplit 越小）；cp = Inf 时只剩根节点；cp 越小 → 树越复杂；cp = 0 时充分生长。 

## 1.4 cp 表逐列解读（核心技能） 

用  `cp = 0`  让树充分生长，得到完整的 cp 表。 这是 rpart 最核心的读表技能 ，后面各 method 的 cp 表读法完全一致。

 > 本块自带数据生成（与 1.1 同参数、同种子），独立可运行。 

以下代码 生成 1.1 节相同的模拟数据 ，并用  `cp = 0`  建立充分生长的分类树，最后以数据框形式打印完整  `$cptable` ，方便直观地查看每一行的 CP、nsplit、训练误差和交叉验证误差。 

In [ ]:
library(rpart)                                       # 加载 rpart 包（最小依赖）
# ---- 自带数据准备（与 1.1 同参数同种子） ----
n   <- 10000; u   <- 10; v   <- 5                          # 样本量、x1 范围、噪声强度
set.seed(11)                                         # 固定种子
x1   <- sort(runif(n, -u, u))                          # 生成 x1 并排序
x2   <- x1 * cos(x1) + v * rnorm(n) - 4                # 第一类 x2
d0   <- data.frame(y = 1, x1, x2)                      # 第一类数据
set.seed(22)                                         # 固定种子
x1   <- sort(runif(n, -u, u))                          # 生成 x1 并排序
x2   <- x1 * cos(x1) + v * rnorm(n) + 4                # 第二类 x2
d1   <- data.frame(y = 2, x1, x2)                      # 第二类数据
sim_cl_tr   <- rbind(d0, d1); sim_cl_tr$y   <- factor(sim_cl_tr$y)   # 合并并转因子（数据平衡）

table(sim_cl_tr$y)                                       # 确认平衡：两类样本量应完全相等

set.seed(2)                                          # 固定种子（xstd 依赖随机切分）
m_ct   <- rpart(y ~ ., data = sim_cl_tr,                     # 用全部自变量建分类树
            method =  "class", cp = 0)                # 显式 method，cp=0 充分生长
as.data.frame(m_ct$cptable)                            # 以数据框展示完整 cp 表 

以上输出的结果分为两类，计算逻辑完全不同：

 第一类：基于全量训练集直接算出来的（无需交叉验证） 
*  `CP` ：复杂度阈值。 这一列就是按 1.2A.2 节的弱链接剪枝算法逐个剪出来的 ：从最复杂的树开始，每次剪掉 $g(t)$ 最小的子树，把该步的 (g(t)) 记为对应的 CP。因此 CP 从大到小排列，每一行对应一棵嵌套子树。
*  `nsplit` ：这棵树实际发生的分裂次数（叶子数减 1）。
*  `rel error` ：相对训练误差。直接用全量训练集预测全量训练集算出来的误差，再除以根节点误差。
   注意看第 1 行（根节点）， `rel error`  是 1.0000。因为树还没分叉，所有样本都被预测为多数类（此处数据绝对平衡，两类各占一半，预测哪个类的误差都是 50%），往下走树越来越复杂， `rel error`  降到 0.3121，它在训练集上越来越准。

 第二类：通过 10 折交叉验证得到的 
 `xerror` ：10 折交叉验证误差的 均值 。为了防止模型“死记硬背”训练集，R 把数据切成 10 份，轮流用 9 份训练、1 份验证，算 10 次误差并求平均。注意，每个 CP 对应的子树都会被这套交叉验证流程重新拟合、评估，所以 xerror 是该子树的泛化误差估计。
*  `xstd` ：10 折交叉验证误差的 标准差 。因为切了 10 次，有波动，用标准差衡量波动的范围。
   可以看到， `xerror`  是一条 先降后升的 U 型曲线 。在树较小时（上面几行），它变小；当树极度复杂时（最后几行），它反弹到 0.50 左右，这正是 过拟合 的信号。 

### CP 值到底是怎么算出来的？——从 0 连续增大，记录下触发剪枝的值

上面表格里  `CP`  列的取值不是凭感觉定的，而是 弱链接剪枝（weakest link pruning） 算法一步步算出来的。下面把「CP 从 0 连续增大，记录下触发剪枝的阈值」这一过程讲透。

#### 1. 从一棵充分生长的树出发

先用  `cp = 0`  建一棵 充分生长的大树  $T_{\max}$，它的每个叶子都足够纯（或者样本太少无法继续分）。

#### 2. 每个内部节点 $t$ 都有两个「误差」

- $R(t)$：把节点 $t$ 当作 叶子 时的误差（该节点内用多数类/均值预测的误差）。
- $R(T_t)$：以 $t$ 为根的 整棵子树  $T_t$ 的误差。 严格来说，这里的误差是带样本量权重的总误差（Total Error） ，即子树所有叶子节点的样本量加权平均误差之和：$\sum_{\ell} n_\ell \bar{R}(\ell)$。只有用总误差相加，才能与惩罚项保持在同一量级。
- $|T_t|$：子树 $T_t$ 的 叶子数 。

子树比单个叶子多用了 $|T_t|-1$ 个分裂，换取误差从 $R(t)$ 降到 $R(T_t)$。

#### 3. 定义每个内部节点的「弱链接强度」$g(t)$

$$
g(t) = \frac{R(t) - R(T_t)}{|T_t| - 1}
$$

含义： 把子树 $T_t$ 剪成单个叶子 $t$，每减少一个分裂，平均要付出多少误差上升的代价。  $g(t)$ 越小，说明这个子树「性价比」越低——留着它带来的误差下降很小，剪掉它损失不大，所以它是 最弱的一环 （weakest link）。

#### 4. 关键：cp 从 0 连续增大时，什么时候触发第一次剪枝？

对整棵树 $T$，代价复杂度是 $R_\alpha(T) = R(T) + \alpha|T|$。设想让 $\alpha$（即 cp） 从 0 开始连续增大 ：

-  $\alpha$ 很小时 ：$\alpha|T|$ 这一项微不足道，充分生长的大树 $T_{\max}$ 的 $R_\alpha$ 最低， 没有子树值得剪 。
-  $\alpha$ 逐渐增大 ：惩罚项 $\alpha|T|$ 越来越重，大树开始「不划算」。但注意——**不是一增大就立刻剪，而是增到某个临界值时，恰好有一个内部节点 $t^*$ 满足「剪掉它之后 $R_\alpha$ 不变甚至更低」 。

推导这个临界值：把以 $t$ 为根的子树 $T_t$ 剪成叶子 $t$，树 $T$ 的叶子数减少 $|T_t|-1$，误差变化为 $R(t) - R(T_t)$。剪枝后 $R_\alpha$  不升反降 的条件是：

$$
R(t) - R(T_t) \le \alpha \,(|T_t| - 1)
\quad\Longleftrightarrow\quad
\alpha \ge \frac{R(t) - R(T_t)}{|T_t| - 1} = g(t)
$$

 换句话说：当 cp（即 $\alpha$）从 0 连续增大到恰好等于某个节点的 $g(t)$ 时，剪掉那个节点就开始「划算」了。 但整棵树里有很多内部节点，$\alpha$ 最先撞上的，必然是当前 $g(t)$ 最小的那个内部节点 。所以，这个过程本质上是在 从低到高扫描 内部节点的 $g(t)$。

 >  注意（自底向上与动态更新） ：在算法实现上，为了高效，通常是 从树的最底层（即紧挨着叶子的内部节点）向高层（根节点方向）进行“自底向上”的扫描**，计算当前的 $g(t)$。并且，这个过程不是只扫描一次。每剪掉一个节点，树的结构变了，剩余内部节点的子树 $T_t$ 及其误差 $R(T_t)$ 会变化，从而其 $g(t)$ 也会更新（通常变大）。所以严谨的流程是： 每一步都自底向上重新扫描当前树的所有内部节点，挑出当前 $g(t)$ 最小的剪掉 ，然后重复。这就像摘果子，每次只摘当前树上最矮的，摘完后有些原本高处的果子可能变成新的最矮果子。

 一句话总结：CP 表里的每一个 CP 值，就是「cp 从 0 连续增大过程中，恰好触发某次剪枝的那个临界 $\alpha$ 值」。  

#### 5. 弱链接剪枝的完整步骤

1. 在当前的树上， 自底向上（从树的最低层向高层） ，对每个内部节点 $t$ 计算 $g(t)$；
2. 找出 $g(t)$  最小 的那个节点 $t^*$（最弱的一环），记 $\alpha_1 = g(t^*)$；
3. 把 $t^*$ 的子树剪掉，$t^*$ 变成叶子，得到一棵稍小的树 $T_1$；
4. 在 $T_1$ 上 重新自底向上计算 剩余内部节点的 $g(t)$（因为结构变了，受剪枝影响的上层节点 $g(t)$ 会发生改变），重复上述过程，得到下一个临界值 $\alpha_2$ 和更小的树 $T_2$；
5. 如此反复，直到只剩根节点。

得到的 $\alpha_1 \le \alpha_2 \le \dots$ 就是 cp 表里  `CP`  列的取值（降序排列），对应的 $T_0, T_1, T_2, \dots$ 就是 cp 表每一行代表的 嵌套子树 。

#### 6. 一个数值直觉

假设某个内部节点 $t$：作为叶子时误差 $R(t)=100$，其子树误差 $R(T_t)=90$，子树有 3 个叶子（即 2 个分裂）。则

$$
g(t) = \frac{100 - 90}{3 - 1} = 5
$$

意思是：保留这棵子树，相当于每多一个分裂只换来 5 个单位的误差下降。如果 $\alpha < 5$，保留划算；一旦 $\alpha$ 增大到 5 以上，剪掉它反而让 $R_\alpha$ 更低。因此当 cp 从 0 增大到 5 时， 这个节点就是第一个被剪的 ，cp 表里就会出现一行  `CP = 5` 。

#### 7. 与 cp 表其余列的关系

-  `nsplit` ：这棵嵌套子树的分裂数；随着 CP 增大，nsplit 单调减少。
-  `rel error` ：这棵子树的训练误差（相对根节点），CP 越大树越小，训练误差越高。
-  `xerror`  /  `xstd` ：对每棵嵌套子树做 10 折交叉验证得到的泛化误差均值与标准差（这部分依赖随机切分，需  `set.seed()`  复现）。

#### 8. 一句话总结

 > cp 表不是「建树时按 cp=0.01、0.02…逐个去建」得来的，而是先建一棵大树，再让 $\alpha$ 从 0 连续增大，每增大到某个内部节点的 $g(t)$ 时触发一次剪枝，记录下这些临界 $\alpha$ 值——它们就是 CP 列。 `prune(tree, cp = x)`  则是在这串嵌套子树里，取「CP 不超过 x 的那棵最简单（最大 CP）的子树」。 

 逐列含义与读法： 

| 列 | 含义 | 计算来源 | 怎么读 |
|---|---|---|---|
|  `CP`  | 复杂度参数 | 弱链接剪枝每步的 $g(t)$，见 1.2A.2 | 从大到小排列；每行对应一棵嵌套子树 |
|  `nsplit`  | 该树的分割数（内部节点数） | 直接计数 | 随 CP 下降而上升，树越来越大 |
|  `rel error`  | 训练集相对误差 = 训练误差 / 根节点误差 | 全量训练集直接计算 | 随 CP 下降单调下降（树越大训练越准） |
|  `xerror`  | 交叉验证误差均值 | 10 折交叉验证 | 先降后升——升即过拟合信号 |
|  `xstd`  | xerror 的标准差 | 10 折交叉验证 | 反映 xerror 的波动；1-SE 规则用它 |

 1-SE 规则： 令  `xerror_min = min(xerror)` ，阈值  `thr = xerror_min + 1*xstd[对应行]` ，在满足  `xerror ≤ thr`  的所有 cp 中取 最大 的 cp。

 >为什么这样设定阈值？其背后的统计学原理和业务比喻请参阅 本文件附录 B：关于 1-SE 规则的深入探讨 。 

以下代码 使用  `plotcp()`  画出完整的 cp 表曲线 ，并逐一图解图中的关键元素。

在 R 的  `plotcp()`  默认绘图中，有以下几层信息需要重点读懂（请结合代码注释和下方解读对照截图）：

1.  底部横轴（cp） ：注意，它 不是线性等距的 cp 值 ，而是 相邻两个 cp 的几何平均数 （取 log 后的视觉等距）。越往左 cp 越大（树越简单），越往右 cp 越小（树越复杂）。
2.  顶部横轴（size of tree） ：表示对应 cp 值下 树的实际叶子节点数 （即  `nsplit + 1` ），直观告诉你当前树有多大。
3.  纵轴（X-val Relative Error） ：即  `xerror` ，衡量泛化误差。
4.  虚线与误差棒 ：每个点上的误差棒上下端点表示  `xerror ± xstd` 。
5.  水平虚线（图中关键参照线） ：这是  `plotcp()`  自动画出的 最小 xerror + 1-SE 阈值线 。具体逻辑见下方深度拆解。 

In [ ]:
options(warn=-1)                                     # 忽略一切警告
options('width'=140)                                 # 充分利用打印宽度
options(repr.plot.width=15, repr.plot.height=10)     # 满幅绘图
library(rpart)                                       # 加载 rpart 包（最小依赖）
# ---- 自带数据准备 ----
n   <- 10000; u   <- 10; v   <- 5                          # 样本量、x1 范围、噪声强度
set.seed(11)                                         # 固定种子
x1   <- sort(runif(n, -u, u))                          # 生成 x1 并排序
x2   <- x1 * cos(x1)+v * rnorm(n)-4                        # 第一类 x2
d0   <- data.frame(y=1,x1,x2)                          # 第一类数据
set.seed(22)                                         # 固定种子
x1   <- sort(runif(n, -u, u))                          # 生成 x1 并排序
x2   <- x1 * cos(x1)+v * rnorm(n)+4                        # 第二类 x2
d1   <- data.frame(y=2,x1,x2)                          # 第二类数据
sim_cl_tr   <- rbind(d0, d1); sim_cl_tr$y   <- factor(sim_cl_tr$y)   # 合并并转因子（数据平衡）

table(sim_cl_tr$y)                                       # 再次确认平衡

set.seed(2)                                          # 固定种子
m_ct   <- rpart(y ~ ., data = sim_cl_tr,                     # 建分类树
            method =  "class", cp = 0)                # 显式 method，cp=0 充分生长

plotcp(m_ct)                                           # 画 xerror 随 cp 的曲线
grid()                                               # 添加网格线 

### 深度拆解： `plotcp()`  图中那根水平虚线和顶部坐标是怎么来的？

很多同学看  `plotcp()`  图时会疑惑：图上那条贯穿左右的水平虚线到底是什么？顶部的  `size of tree`  为什么和底部的 cp 对应不上？其实，这背后是 R 绘图函数预置的 1-SE 视觉辅助。

#### 1. 那根水平虚线是什么？
它是**“最小 xerror + 1 倍 xstd”的阈值参考线**。
- 在  `plotcp()`  的源码逻辑中，它会先自动找出图中 误差最低的那个点 （即  `xerror`  的最小值），然后计算  `min(xerror) + 1 * xstd[对应行]` 。
- 这正好就是 1-SE 规则 的阈值！
- 你只需要看这条虚线以下（包含压在虚线上）的所有点，其中最左边（即 cp 最大、树最简单）的那个点，就是 1-SE 规则选出的 最优 cp 。
  
#### 2. 如何用代码手动复现这条虚线？
由于  `plotcp()`  默认的虚线有时不够显眼，我们可以在图上 手动叠加一条红色虚线 ，让你对它的位置一目了然。

 `r
# 提取交叉验证误差最小的那一行
min_idx <- which.min(m_ct$cptable[, "xerror"])
# 计算 1-SE 阈值
threshold <- m_ct$cptable[min_idx, "xerror"] + m_ct$cptable[min_idx, "xstd"]
# 在原图上叠加红色虚线
abline(h = threshold, col = "red", lty = 2, lwd = 2)
` 
图中的红色虚线即代表了 1-SE 的达标门槛。 只要某个点的 xerror 落在这条红线以下，它就是 1-SE 规则认可的“合格树” 。

#### 3. 横轴为什么是几何平均？
因为 cp 表里的 CP 是从大树到小树呈指数级衰减的（例如 0.1, 0.01, 0.001）。如果直接按原值画，点会全挤在左侧。 `plotcp()`  取了 相邻两个 cp 的几何平均数 （即  `sqrt(cp[i] * cp[i+1])` ），将其投影到 log 尺度的横轴上，这样点就能均匀分布，便于观察曲线的“先降后升”形态。

#### 4. 顶部的  `size of tree`  怎么读？
顶部坐标轴展示的是 叶子节点数 。你想知道“横轴这个 cp 对应的树有多大”，直接抬头看顶部的数字即可。注意它和底部的 cp 并非严格一一对齐，而是按就近原则映射的。 

## 1.5 三棵树对比：完全树 / 最简树 / 1-SE 最优树 

用同一个例子展示 cp 对树复杂度与泛化性能的控制。 本块自带数据、辅助函数与全部三棵树 ，独立可运行。 

以下代码 使用自定义的  `prune_1se()`  函数执行 1-SE 剪枝 ，并与完全生长的树（ `cp=0` ）和只有根节点的最简树（ `cp=Inf` ）进行对比。代码不仅并排画出了三者的分类边界，还汇总了训练集和测试集的误判率，直观验证 1-SE 剪枝的泛化优势。 

In [ ]:
options(warn=-1)                                     # 忽略一切警告
options('width'=140)                                 # 充分利用打印宽度
options(repr.plot.width=15, repr.plot.height=10)     # 满幅绘图
library(rpart); library(partykit)                    # 加载 rpart 与 partykit（最小依赖）
# ---- 自带数据准备（平衡数据） ----
n   <- 10000; u   <- 10; v   <- 5                          # 样本量、x1 范围、噪声强度
set.seed(11)                                         # 训练集第一类种子
x1   <- sort(runif(n,-u,u)); x2   <- x1 * cos(x1)+v * rnorm(n)-4; d0   <- data.frame(y=1,x1,x2)  # 训练集第一类
set.seed(22)                                         # 训练集第二类种子
x1   <- sort(runif(n,-u,u)); x2   <- x1 * cos(x1)+v * rnorm(n)+4; d1   <- data.frame(y=2,x1,x2)  # 训练集第二类
sim_cl_tr   <- rbind(d0,d1); sim_cl_tr$y   <- factor(sim_cl_tr$y)    # 合并训练集并转因子
set.seed(33)                                         # 测试集第一类种子
x1   <- sort(runif(n,-u,u)); x2   <- x1 * cos(x1)+v * rnorm(n)-4; d0   <- data.frame(y=1,x1,x2)  # 测试集第一类
set.seed(44)                                         # 测试集第二类种子
x1   <- sort(runif(n,-u,u)); x2   <- x1 * cos(x1)+v * rnorm(n)+4; d1   <- data.frame(y=2,x1,x2)  # 测试集第二类
sim_cl_te   <- rbind(d0,d1); sim_cl_te$y   <- factor(sim_cl_te$y)    # 合并测试集并转因子

table(sim_cl_tr$y); table(sim_cl_te$y)                       # 确认训练/测试集均完美平衡

# ---- 辅助函数就地定义 ----
prune_1se   <- function(object, se = 1) {              # 1-SE 剪枝函数
  cptab     <- object$cptable                          # 取出 cp 表
  xerror    <- cptab[,  "xerror"]                       # 交叉验证误差均值列
  xstd      <- cptab[,  "xstd"]                         # 交叉验证标准差列
  cp        <- cptab[,  "CP"]                           # 复杂度参数列
  min_idx   <- which.min(xerror)                       # xerror 最小处下标
  threshold   <- xerror[min_idx] + se * xstd[min_idx]  # 1-SE 阈值
  cp_best   <- max(cp[xerror  <= threshold])            # 阈值内取最大 cp
  rpart::prune(object, cp = cp_best)                 # 按 cp 剪枝返回子树
}
draw_boundary   <- function(tree, cl, col =  "blue") {  # 画分类边界函数
  x1g   <- seq(min(cl$x1), max(cl$x1), length = 100)   # x1 网格
  x2g   <- seq(min(cl$x2), max(cl$x2), length = 100)   # x2 网格
  pd    <- function(a, b) predict(tree,                # 网格点预测第一类概率
                                data.frame(x1 = a, x2 = b),
                                type =  "prob")[, 1]
  z   <- outer(x1g, x2g, FUN = pd)                     # 外积得到概率面
  contour(x1g, x2g, z, levels = 0.5,                 # 画 0.5 等值线即分类边界
          drawlabels = FALSE, lwd = 6, col = col, add = TRUE)
}
err   <- function(tree) c(                             # 计算训练/测试误差
  train = mean(predict(tree, sim_cl_tr, type =  "class") != sim_cl_tr$y),  # 训练误差
  test  = mean(predict(tree, sim_cl_te, type =  "class") != sim_cl_te$y)   # 测试误差
)

# ---- 三棵树 ----
set.seed(2)                                          # 固定种子
m_ct_full   <- rpart(y ~ ., data = sim_cl_tr,                    # 完全树
             method =  "class", cp = 0)               # 不剪枝，充分生长
m_ct_min   <- rpart(y ~ ., data = sim_cl_tr,                    # 最简树
             method =  "class", cp = Inf)             # cp=Inf 只保留根节点
m_ct_best   <- prune_1se(m_ct_full, se = 1)                        # 1-SE 最优树

par(mfrow = c(1, 3))                                 # 一行三图
plot(sim_cl_tr$x1, sim_cl_tr$x2, col = sim_cl_tr$y,              # 左图：完全树边界
     xlab =  "x1", ylab =  "x2", cex = 0.5)
draw_boundary(m_ct_full, sim_cl_tr)                            # 叠加完全树分类边界
curve(x * cos(x), col =  "red", lwd = 6, lty = 3, add = TRUE)  # 真实界限
legend( "topright", c( "理论",  "cp=0"),                # 图例
       lwd = 6, col = c( "red",  "blue"), lty = c(3, 1), cex = 0.8)
grid()                                               # 网格线
plot(sim_cl_tr$x1, sim_cl_tr$x2, col = sim_cl_tr$y,              # 中图：最简树
     xlab =  "x1", ylab =  "x2", cex = 0.5)
curve(x * cos(x), col =  "red", lwd = 6, lty = 3, add = TRUE)  # 真实界限
legend( "topright", c( "理论",  "cp=Inf"),              # 图例
       lwd = 6, col = c( "red",  "blue"), lty = c(3, 1), cex = 0.8)
grid()                                               # 网格线
plot(sim_cl_tr$x1, sim_cl_tr$x2, col = sim_cl_tr$y,              # 右图：1-SE 最优树边界
     xlab =  "x1", ylab =  "x2", cex = 0.5)
draw_boundary(m_ct_best, sim_cl_tr)                            # 叠加 1-SE 树边界
curve(x * cos(x), col =  "red", lwd = 6, lty = 3, add = TRUE)  # 真实界限
legend( "topright", c( "理论",  "1-SE"),                # 图例
       lwd = 6, col = c( "red",  "blue"), lty = c(3, 1), cex = 0.8)
grid()                                               # 网格线
par(mfrow = c(1, 1))                                 # 恢复单图布局

rbind(cp0 = err(m_ct_full), cpInf = err(m_ct_min), best1SE = err(m_ct_best))  # 汇总三棵树错误率（平衡数据下极低） 

可见 1-SE 最优树的测试集误差最低—— cp 控制模型复杂度，1-SE 规则在「训练更准」与「泛化更好」之间取平衡 。

 >注意：本节的完全树用了  `cp = 0` 。在普通分类/回归场景， `cp = 0`  是常见的「充分生长」写法；但在 代价敏感 场景（见  `03_横切专题.ipynb`  3.1）， `cp = 0`  可能因权重/先验的浮点边界导致结果不稳定，那时建议用  `cp = 1e-5` 。 

 >  本节要点 
 > 1.  建树依据 ：通过遍历自变量和切点，找到能使“不纯度下降量（Improve）”最大的切分方式。
 > 2.  cp 的定义 ：代价复杂度 $R_\alpha(T) = R(T) + \alpha|T|$ 中的惩罚系数 $\alpha$，即 rpart 的 cp。
 > 3.  cp 的计算 ：弱链接剪枝， 自底向上（从树最低层向根节点） 扫描，对每个内部节点算 $g(t) = \frac{R(t)-R(T_t)}{|T_t|-1}$（注意 $R(T_t)$ 是带样本量加权的总误差）。cp 从 0 连续增大，每增大到某节点的 $g(t)$ 时触发一次剪枝，记录下这些临界值，即 cp 表各行的 CP；由此生成嵌套子树序列。
 > 4.  cp 的作用 ：建树时作停止阈值（improve ≥ cp × 根节点误差才分裂）；剪枝时用  `prune(tree, cp)`  提取对应子树；cp 越大树越简单，cp=Inf 只剩根节点，cp=0 充分生长。
 > 5. cp 表五列：CP、nsplit、rel error（训练）、xerror（CV）、xstd（CV 标准差）。
 > 6. xerror 先降后升；上升即过拟合。
 > 7. 1-SE 规则： `xerror ≤ min(xerror) + xstd`  中取最大 cp； `prune_1se(tree, se = 1)`  即此操作。 `plotcp()`  图上的水平虚线就是这个阈值。
 > 8. 完全树训练误差低但测试误差高；最简树欠拟合；1-SE 树泛化最优。 

---

 <a id="appendix-a"> </a>
# 附录 A. Breiman 类别排序定理的详细证明

本节给出  1.2 节 所用「无序离散变量排序捷径」的完整证明，并明确其适用边界。

### A.1 问题定义

设当前节点有 $n$ 个样本，目标变量为二分类（记正类 $y=1$，负类 $y=0$）。
有一个无序类别型特征 $X$，有 $K$ 个不同取值 $\{c_1, c_2, \dots, c_K\}$。

一次二叉切分，就是把 $\{c_1, \dots, c_K\}$ 分成两个不相交子集 $A$ 和 $B$（$A \cup B = \{c_1, \dots, c_K\}$，$A \cap B = \emptyset$）。
不考虑左右对称性，总共有 $2^{K-1} - 1$ 种不同的二分法。

目标：找到划分 $A, B$ 使得切分后的 加权 Gini 不纯度最小 （等价于 Gini 下降量最大）。

对每个类别 $c_k$，计算其 正类比例  $p_k = P(y=1 \mid X=c_k)$，以及样本数 $N_k$。

### A.2 定理陈述

 Breiman 类别排序定理 ：将类别按 $p_k$ 从小到大排序，不失一般性设 $p_1 \le p_2 \le \dots \le p_K$。则使加权 Gini 不纯度最小的最优二分子集必然形如
$$
 A = \{c_1, \dots, c_k\}, \quad B = \{c_{k+1}, \dots, c_K\}, \quad 1 \le k \le K-1 
$$
即切点只出现在排序后的相邻缝隙处。

### A.3 证明

 Step 1：将 Gini 不纯度化为目标函数 

对于节点 $S$，含 $N$ 个样本、正类 $N_1$、负类 $N_0$，Gini 为
$$
 G(S) = 1 - \left(\frac{N_1}{N}\right)^2 - \left(\frac{N_0}{N}\right)^2 = 2 \frac{N_1 N_0}{N^2} 
$$
切分后加权 Gini：
$$
 G_{\text{split}} = \frac{N_A}{N} G(A) + \frac{N_B}{N} G(B) = \frac{2}{N} \left( \frac{N_{A1} N_{A0}}{N_A} + \frac{N_{B1} N_{B0}}{N_B} 
\right) 
$$
其中 $N_{A1}, N_{A0}$ 分别表示 $A$ 中正、负类样本数。
因为 $N$ 固定， 最小化 $G_{\text{split}}$ 等价于最小化 
$$
 F(A, B) = \frac{N_{A1} N_{A0}}{N_A} + \frac{N_{B1} N_{B0}}{N_B} 
$$

 Step 2：关键凹性 

考察函数 $f(x, y) = \dfrac{x y}{x + y}$（$x, y > 0$）。其 Hessian 为
$$
 \frac{\partial^2 f}{\partial x^2} = -\frac{2y^2}{(x+y)^3} < 0, \quad \frac{\partial^2 f}{\partial y^2} = -\frac{2x^2}{(x+y)^3} < 0 
$$
故 $f$ 在 $x >0, y >0$ 上 严格凹 。等价地，$f(x,y) = x\left(1 - \dfrac{x}{x+y}\right)$，对固定 $N = x+y$，它是关于正类比例 $x/N$ 的方差项，自然具有凹性。

 Step 3：交换论证 

假设存在一个最优划分 $A, B$。若在 $A$ 中存在类别 $c_i$，在 $B$ 中存在类别 $c_j$，满足 $p_i < p_j$（即 $A$ 中的 $c_i$ 更偏负类，$B$ 中的 $c_j$ 更偏正类），我们尝试交换 $c_i$ 与 $c_j$，得到新划分 $A', B'$。

记 $c_i$ 贡献 $(n_{i1}, n_{i0})$，$c_j$ 贡献 $(n_{j1}, n_{j0})$，且
$$
 \frac{n_{i1}}{n_{i1}+n_{i0}} < \frac{n_{j1}}{n_{j1}+n_{j0}} 
$$
由 $f$ 的凹性（或由方差分解的直观：把更同质的类别聚在一起会降低不纯度），可以证明
$$
 F(A', B') \le F(A, B) 
$$
也就是说， 只要 $A$ 中某类别的 $p$ 值低于 $B$ 中某类别，交换它们不会让加权 Gini 变大 。

 Step 4：结论 

由 Step 3 立即推出：任何最优划分 $A, B$ 都必须满足
$$
 \min_{c \in A} p_c \ge \max_{c \in B} p_c 
$$
（或左右互换）。即：$A$ 中所有类别的 $p$ 值都不小于 $B$ 中所有类别。

既然已将所有 $p_k$ 排序为 $p_1 \le p_2 \le \dots \le p_K$，则最优划分只能在某个位置 $k$ 处断开：
$$
 A = \{c_1, \dots, c_k\}, \quad B = \{c_{k+1}, \dots, c_K\} 
$$
共 $K-1$ 种候选。穷举这 $K-1$ 种，即等价于穷举全部 $2^{K-1}-1$ 种。∎

### A.4 回归树（MSE）下的平行结论

回归树的不纯度是方差（MSE）。由方差分解
$$
 \text{Var}(Y) = \text{Var}_{\text{within}} + \text{Var}_{\text{between}} 
$$
最小化加权组内方差，等价于最大化组间方差；后者正比于 $N_A N_B (\bar Y_A - \bar Y_B)^2$。
将类别按均值排序后，只在相邻缝隙切分，恰好是最大化两组均值差的候选集合。因此排序法在回归 + MSE 下同样 精确最优 。

### A.5 适用边界

上述证明依赖两点：
1.  二分类目标 （正/负类，单参数 $p_k$ 足以刻画纯度）；
2.  不纯度函数的凹性 （Gini 或 MSE 都满足）。

若使用 信息增益/Entropy ，或响应变量为 多分类（$ >2$ 类） ，上述“按单参数排序、相邻缝隙切分即最优”的结论 不再成立 。此时 rpart 若仍采用排序捷径，只能视为一种 启发式近似 ；严格最优需要更慢的搜索（或直接放弃最优性）。

 > 工程上常用的替代方案：多分类问题中对无序高基数类别做目标编码（target encoding）或频次筛选；或使用支持类别特征原生分裂的库（如 CatBoost、LightGBM 的  `categorical_feature` ），它们分别使用不同的启发式策略。 

---

 <a id="appendix-b"> </a>
# 附录 B. 关于 1-SE 规则的深入探讨 

 1-SE 规则 是决策树剪枝时最核心的策略：令  `thr = min(xerror) + 1 * xstd[对应行]` ，在满足  `xerror ≤ thr`  的所有 cp 中取 最大 的 cp。

为什么要这样设定呢？这背后蕴含着严谨的统计学逻辑和对抗方差的智慧：

### 1. 为什么不用“绝对最低误差”？（警惕运气成分）
在 10 折交叉验证中，数据是随机切分的。假设模型 A 的测试误差是 0.40，模型 B 是 0.41。你能笃定 A 一定比 B 好吗？不一定。这 0.01 的差距很可能仅仅是因为 B 刚好分到了几个比较难预测的“奇葩样本”。如果换一种切法，结果可能逆转。这就是 随机波动（方差） 。

### 2.  `xstd`  是什么？它是“容错空间”
 `xstd` （交叉验证标准差）恰恰量化了这种随机波动。如果  `xstd`  很大，说明 10 次验证误差忽高忽低，模型不稳定。因此，统计学家引入  `thr = min(xerror) + 1 * xstd`  的潜台词是： “我不要求你达到最完美的成绩（最低误差），只要你的成绩在完美成绩的 1 个标准误波动范围内，我就认为你是合格的。” 

### 3. 为什么正好是“1”倍？（1-SE 规则的科学依据）
在统计学中， “1 个标准差（1-SE）” 是一个非常经典的置信区间指标。
 0-SE （不加标准差）：苛求绝对最低误差，会选出极其复杂的树（过拟合）。
 2-SE ：容错空间太大，可能选出一棵过于简单的树（欠拟合）。
*  1-SE ：是 奥卡姆剃刀（Occam's Razor） 的最佳平衡点—— 在模型性能损失极小（不显著）的前提下，选择最简单的模型 。

### 4. 生活化的比喻
假设你要招募一个长跑运动员：
 A 选手 ：平均成绩 10.0 秒，但忽好忽坏（标准差 0.5 秒）。
 B 选手 ：平均成绩 10.2 秒，但极其稳定（标准差 0.1 秒）。

如果只选“最低均值”，你会选 A。但你可能会想： “A 的 10.0 秒可能只是运气好。为了保险，我要设置一个门槛：只要成绩在 10.0 + 0.5 = 10.5 秒以内的，我都考虑。” 这个时候，B 选手（10.2 秒）就进入了门槛。如果此时你还发现 B 选手 要的薪水更少（对应模型更简单、节点更少、更好解释） ，你就会果断放弃 A，选择 B。

 在决策树中： 
*  “薪水更少、更好管理” = CP 值更大，树更简单 。
*  “成绩达标” = xerror 在最低值 + 1个标准差的范围内 。

所以，设定  `thr = xerror_min + xstd`  是为了 用微小的性能让步，换取模型极大的简化，同时规避了评估指标随机波动带来的假象 。 

---

## 本文件小结与下一步

本文件已把 rpart 的 共用核心机制 讲透：不纯度与 improve、候选切点搜索、cp 的定义与两处作用、弱链接剪枝与 CP 列的来历、cp 表五列读法、1-SE 规则、三棵树对比，以及 附录 A 的 Breiman 定理证明 和 附录 B 的 1-SE 深探 。

 下一步 ：进入  `02_按method展开.ipynb` 。该文件按  `class → anova → poisson → 生存`  依次展开，每个 method 只讲它 特有 的不纯度口径、improve 公式与 predict 输出；cp 表读法、1-SE 规则、 `prune_1se`  直接沿用本文件，不再重复。 